# Week 3 - SQL Queries & Performance Dashboard
**Objective:** Query relational tables using SQLite and analyze program performance metrics.
```

In [ ]:
import sqlite3
import pandas as pd

# Connect to (or create) the SQLite database
conn = sqlite3.connect('nextgen.db')
cursor = conn.cursor()

# 1. Create applicants table
cursor.execute('''
CREATE TABLE IF NOT EXISTS applicants (
    applicant_id INTEGER PRIMARY KEY,
    name TEXT,
    domain TEXT,
    university TEXT,
    application_date DATE,
    status TEXT
);
''')

# 2. Create interns table (connects via applicant_id)
cursor.execute('''
CREATE TABLE IF NOT EXISTS interns (
    intern_id INTEGER PRIMARY KEY,
    applicant_id INTEGER,
    domain TEXT,
    start_date DATE,
    completion_status TEXT,
    FOREIGN KEY (applicant_id) REFERENCES applicants(applicant_id)
);
''')

# 3. Create hackathon_scores table (connects via intern_id)
cursor.execute('''
CREATE TABLE IF NOT EXISTS hackathon_scores (
    intern_id INTEGER PRIMARY KEY,
    score INTEGER,
    domain TEXT,
    FOREIGN KEY (intern_id) REFERENCES interns(intern_id)
);
''')

# Insert sample data to populate the tables
cursor.executescript('''
DELETE FROM hackathon_scores;
DELETE FROM interns;
DELETE FROM applicants;

INSERT INTO applicants VALUES
(1, 'Alice Smith', 'Web Development', 'FAST', '2026-05-10', 'Selected'),
(2, 'Bob Jones', 'Web Development', 'NUST', '2026-05-12', 'Selected'),
(3, 'Charlie Brown', 'Web Development', 'GIKI', '2026-05-14', 'Rejected'),
(4, 'David Miller', 'Cloud Computing', 'NUST', '2026-05-15', 'Selected'),
(5, 'Emma Wilson', 'Cloud Computing', 'LUMS', '2026-05-16', 'Selected'),
(6, 'Frank Davis', 'Cybersecurity', 'UET', '2026-05-17', 'Selected'),
(7, 'Grace Lee', 'Cybersecurity', 'FAST', '2026-05-18', 'Rejected'),
(8, 'Hannah Taylor', 'AI & Data Science', 'NUST', '2026-05-19', 'Selected'),
(9, 'Ian Anderson', 'AI & Data Science', 'LUMS', '2026-05-20', 'Selected'),
(10, 'Julia White', 'Video Editing', 'GIKI', '2026-05-21', 'Selected');

INSERT INTO interns VALUES
(101, 1, 'Web Development', '2026-06-01', 'Completed'),
(102, 2, 'Web Development', '2026-06-01', 'Dropped Out'),
(103, 4, 'Cloud Computing', '2026-06-01', 'Completed'),
(104, 5, 'Cloud Computing', '2026-06-01', 'Completed'),
(105, 6, 'Cybersecurity', '2026-06-01', 'Completed'),
(106, 8, 'AI & Data Science', '2026-06-01', 'Completed'),
(107, 9, 'AI & Data Science', '2026-06-01', 'Dropped Out'),
(108, 10, 'Video Editing', '2026-06-01', 'Completed');

INSERT INTO hackathon_scores VALUES
(101, 88, 'Web Development'),
(103, 92, 'Cloud Computing'),
(104, 84, 'Cloud Computing'),
(105, 78, 'Cybersecurity'),
(106, 95, 'AI & Data Science'),
(108, 70, 'Video Editing');
''')

conn.commit()
print("Database and tables successfully initialized!")

Database and tables successfully initialized!


## Step 2: Exploring the Relational Tables
Inspecting the first 5 rows of each table to understand column names and schema structures before querying.

In [ ]:
print("--- Applicants (Top 5) ---")
print(pd.read_sql("SELECT * FROM applicants LIMIT 5;", conn))

print("\n--- Interns (Top 5) ---")
print(pd.read_sql("SELECT * FROM interns LIMIT 5;", conn))

print("\n--- Hackathon Scores (Top 5) ---")
print(pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;", conn))

--- Applicants (Top 5) ---
   applicant_id           name           domain university application_date  \
0             1    Alice Smith  Web Development       FAST       2026-05-10   
1             2      Bob Jones  Web Development       NUST       2026-05-12   
2             3  Charlie Brown  Web Development       GIKI       2026-05-14   
3             4   David Miller  Cloud Computing       NUST       2026-05-15   
4             5    Emma Wilson  Cloud Computing       LUMS       2026-05-16   

     status  
0  Selected  
1  Selected  
2  Rejected  
3  Selected  
4  Selected  

--- Interns (Top 5) ---
   intern_id  applicant_id           domain  start_date completion_status
0        101             1  Web Development  2026-06-01         Completed
1        102             2  Web Development  2026-06-01       Dropped Out
2        103             4  Cloud Computing  2026-06-01         Completed
3        104             5  Cloud Computing  2026-06-01         Completed
4        105       

### Query 1: How many interns completed each domain's program?
Counts completed interns grouped by domain to see which tracks had the highest number of successful finishers.

In [ ]:
# Query 1: Counts completed interns grouped by domain
query1 = """
SELECT domain, COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""
df1 = pd.read_sql(query1, conn)
print(df1)

              domain  completed_count
0    Cloud Computing                2
1    Web Development                1
2      Video Editing                1
3      Cybersecurity                1
4  AI & Data Science                1


### Query 2: What is the average hackathon score per domain?
Calculates the mean hackathon score within each domain, rounded to two decimal places.

In [ ]:
# Query 2: Calculates the average hackathon score for each domain
query2 = """
SELECT domain, ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""
df2 = pd.read_sql(query2, conn)
print(df2)

              domain  avg_score
0  AI & Data Science       95.0
1    Web Development       88.0
2    Cloud Computing       88.0
3      Cybersecurity       78.0
4      Video Editing       70.0


### Query 3: Which interns scored above a threshold (top performers)?
Joins interns with hackathon scores to list individuals scoring 85 or above for certificate/showcase eligibility.

In [ ]:
# Query 3: Lists interns who scored 85 or above
query3 = """
SELECT i.intern_id, i.domain, h.score
FROM interns i
JOIN hackathon_scores h ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""
df3 = pd.read_sql(query3, conn)
print(df3)

   intern_id             domain  score
0        106  AI & Data Science     95
1        103    Cloud Computing     92
2        101    Web Development     88


### Query 4: What is the conversion rate from "applied" to "completed" per domain?
Uses a LEFT JOIN to compare total applicants against completed interns to evaluate domain conversion percentages accurately.

In [ ]:
# Query 4: Compares total applicants vs completed interns per domain to find conversion rate
query4 = """
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id), 2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
    ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""
df4 = pd.read_sql(query4, conn)
print(df4)

              domain  total_applicants  total_completed  conversion_rate_pct
0      Video Editing                 1                1               100.00
1    Cloud Computing                 2                2               100.00
2      Cybersecurity                 2                1                50.00
3  AI & Data Science                 2                1                50.00
4    Web Development                 3                1                33.33


### Query 5: How many interns dropped out vs completed per domain?
Aggregates retention outcomes per domain to identify which fields experienced the highest dropout rates.

In [ ]:
# Query 5: Measures retention by counting completed vs dropped out interns per domain
query5 = """
SELECT
    domain,
    SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END) AS completed_count,
    SUM(CASE WHEN completion_status = 'Dropped Out' THEN 1 ELSE 0 END) AS dropped_out_count
FROM interns
GROUP BY domain
ORDER BY completed_count DESC;
"""
df5 = pd.read_sql(query5, conn)
print(df5)

              domain  completed_count  dropped_out_count
0    Cloud Computing                2                  0
1    Web Development                1                  1
2      Video Editing                1                  0
3      Cybersecurity                1                  0
4  AI & Data Science                1                  1


In [ ]:
!pip install streamlit -q
!npm install -g localtunnel -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [ ]:
%%writefile dashboard.py
import sqlite3
import pandas as pd
import streamlit as st

st.set_page_config(page_title="NextGenLearners Dashboard", layout="wide")
st.title("NextGenLearners Program Performance Dashboard")

# Connect to the SQLite database
conn = sqlite3.connect('nextgen.db')
applicants = pd.read_sql("SELECT * FROM applicants;", conn)
interns = pd.read_sql("SELECT * FROM interns;", conn)
scores = pd.read_sql("SELECT * FROM hackathon_scores;", conn)

# Interactive Domain Filter (Required)
domain_list = ['All'] + sorted(applicants['domain'].unique().tolist())
selected_domain = st.selectbox('Filter by Domain', domain_list)

# Filter DataFrames based on dropdown selection
if selected_domain != 'All':
  app_filt = applicants[applicants['domain'] == selected_domain]
  int_filt = interns[interns['domain'] == selected_domain]
  sco_filt = scores[scores['domain'] == selected_domain]
else:
  app_filt = applicants
  int_filt = interns
  sco_filt = scores

# Element 1: Funnel (Total Applicants vs Completed Interns)
st.subheader("1. Funnel Overview")
col1, col2 = st.columns(2)
completed_count = len(int_filt[int_filt['completion_status'] == 'Completed'])
col1.metric("Total Applicants", len(app_filt))
col2.metric("Completed Interns", completed_count)

st.markdown("---")

# Elements 2 & 3: Completion Rate and Average Score Charts
col3, col4 = st.columns(2)

with col3:
  st.subheader("2. Completion Rate per Domain (%)")
  conv_df = pd.read_sql(
      """
    SELECT
        a.domain,
        ROUND(100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id), 1) AS completion_rate_pct
    FROM applicants a
    LEFT JOIN interns i
        ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
    GROUP BY a.domain;
    """,
      conn,
  )
  if selected_domain != 'All':
    conv_df = conv_df[conv_df['domain'] == selected_domain]
  st.bar_chart(conv_df.set_index('domain'))

with col4:
  st.subheader("3. Average Hackathon Score per Domain")
  avg_score_df = pd.read_sql(
      """
    SELECT domain, ROUND(AVG(score), 1) AS avg_score
    FROM hackathon_scores
    GROUP BY domain;
    """,
      conn,
  )
  if selected_domain != 'All':
    avg_score_df = avg_score_df[avg_score_df['domain'] == selected_domain]
  st.bar_chart(avg_score_df.set_index('domain'))

st.markdown("---")

# Element 4: Leaderboard of Top 10 Performers
st.subheader("4. Top Performers Leaderboard")
leaderboard = pd.read_sql(
    """
SELECT i.intern_id, a.name, i.domain, h.score
FROM interns i
JOIN applicants a ON i.applicant_id = a.applicant_id
JOIN hackathon_scores h ON i.intern_id = h.intern_id
ORDER BY h.score DESC
LIMIT 10;
""",
    conn,
)
if selected_domain != 'All':
  leaderboard = leaderboard[leaderboard['domain'] == selected_domain]
st.dataframe(leaderboard, use_container_width=True)

Writing dashboard.py


In [ ]:
import urllib.request

# 1. Print the tunnel password IP
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("Your Tunnel Password IP is:", ip)

# 2. Run Streamlit and expose via Localtunnel
!streamlit run dashboard.py & npx localtunnel --port 8501

Your Tunnel Password IP is: 8.228.117.222
⠙

⠹⠸⠼⠴your url is: https://curly-turkeys-taste.loca.lt
2026-11-06 15:31:56.336 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.117.222:8501

2026-11-06 15:34:27.361 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...
^C


In [13]:
# 1. Download Cloudflare tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Run Streamlit with Cloudflare tunnel
import subprocess
import time

subprocess.Popen(["streamlit", "run", "dashboard.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)

# 3. Expose the port and print the public link
!cloudflared tunnel --url http://localhost:8501

2026-11-06T15:36:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-06T15:36:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-06T15:36:42Z INF +--------------------------------------------------------------------------------------------+
2026-11-06T15:36:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-06T15:36:42Z INF |  https://freeze-doug-volume-oils.trycloudflare.com    